In [ ]:
!pip install -q open_clip_torch onnx onnxruntime

In [ ]:
import os
import numpy as np
import torch
import open_clip
import onnx
import onnxruntime as ort

device = "cuda" if torch.cuda.is_available() else "cpu"

# 加载与 baseline 相同的模型
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32-quickgelu",
    pretrained="openai"
)

model = model.to(device)
model.eval()

print("设备:", device)
print("模型加载完成")

In [ ]:
%pip install -q onnx onnxscript onnxruntime

In [ ]:
import torch.nn.functional as F

class ImageEncoderONNX(torch.nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.clip_model = clip_model

    def forward(self, image):
        features = self.clip_model.encode_image(image)
        return F.normalize(features, dim=-1)

onnx_model = ImageEncoderONNX(model).to(device)
onnx_model.eval()

# 固定输入尺寸：[batch, 3, 224, 224]
dummy_image = torch.randn(
    1, 3, 224, 224,
    device=device
)

onnx_path = "/content/clip_image_encoder.onnx"

with torch.no_grad():
    torch.onnx.export(
        onnx_model,
        dummy_image,
        onnx_path,
        input_names=["image"],
        output_names=["image_features"],
        dynamic_axes={
            "image": {0: "batch"},
            "image_features": {0: "batch"},
        },
        opset_version=17,
    )

print("ONNX 已导出:", onnx_path)
print("文件大小 MB:", os.path.getsize(onnx_path) / 1024 / 1024)

In [ ]:
# 检查模型结构
onnx_model_check = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model_check)
print("ONNX 模型检查通过")

# 使用 ONNX Runtime CPU 推理
session = ort.InferenceSession(
    onnx_path,
    providers=["CPUExecutionProvider"]
)

dummy_numpy = dummy_image.cpu().numpy()

onnx_output = session.run(
    None,
    {"image": dummy_numpy}
)[0]

with torch.no_grad():
    torch_output = onnx_model(dummy_image).cpu().numpy()

print("PyTorch 输出:", torch_output.shape)
print("ONNX 输出:", onnx_output.shape)
print(
    "最大绝对误差:",
    np.max(np.abs(torch_output - onnx_output))
)

In [ ]:
np.save("/content/onnx_dummy_image.npy", dummy_numpy)
np.save("/content/onnx_dummy_output.npy", onnx_output)

print("已保存测试输入输出")

In [ ]:
from google.colab import files

files.download("/content/clip_image_encoder.onnx")
files.download("/content/onnx_dummy_image.npy")
files.download("/content/onnx_dummy_output.npy")